# Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
from itertools import product

In [ ]:
import matplotlib as mpl

plt.style.use('dark_background')
mpl.rcParams['legend.frameon'] = False

In [ ]:
# define folder to output generated figures
path_figures = Path('./figures')
path_figures.mkdir(exist_ok=True)

# define figure parameters
DPI=200

def save_figure(filename):
    plt.savefig(f"{path_figures}/{filename}.webp", dpi=DPI, transparent=True)

# Computing dice throw distributions

In [ ]:
def compute_distribution(N_faces, N_dice):
    """
    Compute the distribution of outcomes for N_faces x N_dice
    returns (list_values, list_probabilities)
    """
    # list of possible outcomes for one throw
    throw_outcomes = list(range(1, N_faces+1))

    # generator containing all possible throws (with multiple dice)
    all_outcomes = product(throw_outcomes, repeat=N_dice)

    # compute sum total for each throw 
    list_results = [sum(i) for i in all_outcomes]

    # compute how many times each total sum is achieved
    values, counts = np.unique(list_results, return_counts=True)

    mean = np.mean(list_results)
    sd = np.std(list_results)

    return values, counts, mean, sd

In [ ]:
N_faces = 6
N_dice = 2

In [ ]:
compute_distribution(N_faces, N_dice)

# Visualization

## Array of distributions

In [ ]:
min_faces, max_faces = 2, 8
min_dice, max_dice = 1, 8

dict_data = {}
for i, i_face in enumerate(range(min_faces, max_faces+1)):
    for j, j_dice in enumerate(range(min_dice, max_dice+1)):  
        value, counts, mean, std = compute_distribution(N_faces=i_face, N_dice=j_dice)
        idx = (j_dice, i_face)
        dict_data[idx] = [j_dice, i_face, mean, std, std/mean]

In [ ]:
min_faces, max_faces = 2, 6
min_dice, max_dice = 1, 6

fig, axarr = plt.subplots(nrows=max_faces-min_faces+1, ncols=max_dice-min_dice+1, figsize=(12,10), dpi=DPI)

for i, i_face in enumerate(range(min_faces, max_faces+1)):
    for j, j_dice in enumerate(range(min_dice, max_dice+1)):
        
        value, counts, mean, std = compute_distribution(N_faces=i_face, N_dice=j_dice)

        ax = axarr[i,j]
        ax.bar(x=value, height=counts/sum(counts))

        ax.set_title(f"N={sum(counts)}\n$\\mu$={mean:.2f} $\\frac{{\\sigma}}{{\\mu}}$={std/mean:.3f}", fontsize=14)
        ax.set_yticks([])

plt.tight_layout()
save_figure("array_histograms")

## Comparing distributions

In [ ]:
df_throws = pd.DataFrame(dict_data.values(), columns=["N_dice", "N_faces", "mean", "std", "std_norm"])
df_throws.head(20)

In [ ]:
fig = plt.figure(dpi=DPI)
for i, group in df_throws.groupby('N_faces'):
    plt.plot(group['mean'], group['std'], label=str(i))
    plt.scatter(group['mean'], group['std'])
plt.legend()
plt.ylabel('Std')
plt.xlabel('Mean')
plt.tight_layout()
save_figure("dist_props_dice_vs_nfaces")

In [ ]:
fig = plt.figure(dpi=DPI)
for i, group in df_throws.groupby('N_faces'):
    plt.plot(group['mean'], group['std_norm'], label=str(i))
    plt.scatter(group['mean'], group['std_norm'])
plt.legend()
plt.ylabel('Std / Mean')
plt.xlabel('Mean')
plt.tight_layout()
save_figure("dist_norm_props_dice_vs_nfaces")

In [ ]:
fig = plt.figure(dpi=DPI)
for n_dice in range(2, 9):
    value, counts, mean, std = compute_distribution(N_faces=6, N_dice=n_dice)
    plt.plot(value, counts/sum(counts), label=str(n_dice))
plt.legend()
plt.yticks([])
plt.tight_layout()
save_figure("dists_by_ndice")

## Distributions with the same mean 

In [ ]:
target_df = df_throws.groupby('mean').filter(lambda x: len(x) > 2)
target_df

In [ ]:
fig, axarr = plt.subplots(ncols=3, figsize=(10,4))

for i, (mean, temp_df) in enumerate(target_df.groupby('mean')):
    ax = axarr[i]
    for j, row in temp_df.iterrows():
        n_dice = row['N_dice'].astype(int)
        n_faces = row['N_faces'].astype(int)
    
        value, counts, mean, std = compute_distribution(N_faces=n_faces, N_dice=n_dice)
        ax.plot(value, counts/sum(counts), label=f"{n_dice}d{n_faces}")
    ax.legend(fontsize=14)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title(f"Distributions with mean={mean}", fontsize=14)
    
plt.tight_layout()
save_figure(f"dists_centered_means")